#set-up

In [ ]:
!nvidia-smi || true

# diffusers 0.34.0 matches the script; gradio is for the UI; OpenCV is for Canny edge detection; the rest are common dependencies
!pip -q install -U diffusers==0.34.0 transformers accelerate safetensors \
  opencv-python pillow gradio

# Optional: if you run into OpenCV libGL issues, run the line below
# !apt -yq install libgl1


Thu Aug 14 00:35:47 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

#ControlNet + SDXL

In [ ]:
import os, traceback
from typing import Optional, Tuple
import numpy as np
from PIL import Image
import torch, gradio as gr, cv2

from diffusers import (
    ControlNetModel,
    StableDiffusionXLControlNetPipeline,
    AutoencoderKL,
)

# ============== Device & Precision ==============

def pick_device_dtype() -> Tuple[torch.device, torch.dtype]:
    if torch.cuda.is_available():
        return torch.device("cuda"), torch.float16
    return torch.device("cpu"), torch.float32

DEVICE, DTYPE = pick_device_dtype()

# ============== Models ==============
class ModelBundle:
    def __init__(self):
        self.vae = None
        self.pipe_canny = None
        self._ip_loaded = False  # whether IP-Adapter is loaded

    def load_vae(self):
        if self.vae is None:
            self.vae = AutoencoderKL.from_pretrained(
                "madebyollin/sdxl-vae-fp16-fix",
                torch_dtype=DTYPE,
            )
        return self.vae

    def _configure(self, pipe):
        if DEVICE.type == "cuda":
            pipe.enable_model_cpu_offload()  # save VRAM
            pipe.enable_vae_slicing()
            pipe.enable_vae_tiling()
        else:
            pipe.enable_sequential_cpu_offload()
        pipe.set_progress_bar_config(disable=True)

    def load_canny_pipe(self):
        if self.pipe_canny is None:
            cn = ControlNetModel.from_pretrained(
                "diffusers/controlnet-canny-sdxl-1.0",
                torch_dtype=DTYPE,
            )
            self.pipe_canny = StableDiffusionXLControlNetPipeline.from_pretrained(
                "stabilityai/stable-diffusion-xl-base-1.0",
                controlnet=cn,
                vae=self.load_vae(),
                torch_dtype=DTYPE,
            )
            self._configure(self.pipe_canny)
        return self.pipe_canny

    def ensure_ip_adapter(self, pipe) -> str:
        """
        Load IP-Adapter (color/style reference) for the pipeline.
        Also move the image_encoder to the same device/dtype as inference
        to avoid HalfTensor device/dtype mismatch.
        """
        try:
            if not getattr(self, "_ip_loaded", False):
                pipe.load_ip_adapter(
                    "h94/IP-Adapter",
                    subfolder="sdxl_models",
                    weight_name="ip-adapter_sdxl.bin",
                )
                self._ip_loaded = True

            # Key fix: ensure image_encoder is on the same device & dtype
            if hasattr(pipe, "image_encoder") and pipe.image_encoder is not None:
                try:
                    pipe.image_encoder.to(DEVICE, dtype=DTYPE)
                except Exception:
                    # Some offload combos may raise on .to; fallback: move to CPU then back
                    pipe.image_encoder.to("cpu")
                    pipe.image_encoder.to(DEVICE, dtype=DTYPE)

            return "ok"
        except Exception as e:
            return f"IP-Adapter load failed: {e}"

MODELS = ModelBundle()

# ============== Preprocess ==============

def to_rgb(img: Image.Image) -> Image.Image:
    if img is None:
        return None
    if img.mode != "RGB":
        img = img.convert("RGB")
    return img


def make_canny(image: Image.Image, low: int, high: int, size: Tuple[int, int]) -> Image.Image:
    """
    Create a Canny control image resized to the target resolution to avoid size mismatch.
    """
    image = to_rgb(image).resize(size, Image.BICUBIC)
    a = np.array(image)
    edges = cv2.Canny(a, low, high)
    edges = np.repeat(edges[:, :, None], 3, axis=2)
    return Image.fromarray(edges)

# ============== Generate ==============

def generate(
    prompt: str,
    negative_prompt: str,
    control_image: Image.Image,
    height: int,
    width: int,
    steps: int,
    guidance: float,
    control_scale: float,
    canny_low: int,
    canny_high: int,
    seed: Optional[int],
    use_ref: bool,
    ref_image: Image.Image,
    ref_strength: float,
):
    try:
        if control_image is None:
            return None, "Please upload a control image first."

        # Normalize resolution → build Canny
        size = (int(width), int(height))
        control = make_canny(control_image, canny_low, canny_high, size)

        pipe = MODELS.load_canny_pipe()

        # Seed
        generator = None
        if seed not in (None, ""):
            seed = int(seed)
            generator = torch.Generator(device=DEVICE).manual_seed(seed)

        # Optional color reference: when enabled, load IP-Adapter and pass ref image
        extra = ""
        ip_kwargs = {}
        if use_ref:
            status = MODELS.ensure_ip_adapter(pipe)
            if status == "ok":
                pipe.set_ip_adapter_scale(float(ref_strength))  # 0~1
                ip_kwargs["ip_adapter_image"] = to_rgb(ref_image) if ref_image else to_rgb(control_image)
                extra = " / color reference: ON"
            else:
                extra = f" / color reference: OFF ({status})"

        # Inference
        with torch.inference_mode():
            image = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt if negative_prompt else None,
                num_inference_steps=int(steps),
                guidance_scale=float(guidance),
                controlnet_conditioning_scale=float(control_scale),
                image=control,                   # ControlNet input
                height=size[1], width=size[0],   # keep same as control image
                generator=generator,
                **ip_kwargs,
            ).images[0]

        return image, f"Done (device: {DEVICE.type}{extra})"

    except torch.cuda.OutOfMemoryError:
        return None, "CUDA OOM: please reduce resolution to 768×768 or steps to 20–25 and try again."
    except Exception as e:
        # Return full traceback to UI for easier debugging
        tb = traceback.format_exc()
        return None, f"Error occurred: {e}\n\nDetails:\n{tb}"

# ============== Gradio UI ==============
with gr.Blocks(title="SDXL + ControlNet (minimal)") as demo:
    gr.Markdown(
        "## 🧨 SDXL + ControlNet (lean & stable)\n"
        "Canny structural control + optional IP-Adapter color/style reference.\n"
        "If an error occurs, the full traceback will appear below."
    )

    with gr.Row():
        with gr.Column(scale=1):
            control_image = gr.Image(type="pil", label="Control image (any image)")
            prompt = gr.Textbox(
                label="Prompt",
                value=(
                    "small piles of debris along roadside and in a docking area, paved and dirt roads, "
                    "elevated houses along the road, upright telephone poles, green leafy trees, "
                    "coastal small town after mild hurricane, calm clean atmosphere, houses structurally sound, "
                    "realistic lighting, high detail, ultra realistic, 8k"
                ),
            )
            negative_prompt = gr.Textbox(
                label="Negative prompt",
                value=(
                    "severe destruction, collapsed buildings, broken telephone poles, uprooted trees, "
                    "flooded streets, heavy storm damage, chaotic scene, ruined houses, fire, smoke, "
                    "low quality, blurry, distorted, oversaturated, lowres, artifacts"
                ),
            )
            seed = gr.Textbox(label="Seed (optional)", placeholder="e.g. 42")

        with gr.Column(scale=1):
            height = gr.Slider(512, 1280, value=768, step=64, label="Height")
            width = gr.Slider(512, 1280, value=768, step=64, label="Width")
            steps = gr.Slider(10, 60, value=30, step=1, label="Steps")
            guidance = gr.Slider(1.0, 12.0, value=6.0, step=0.1, label="CFG (text adherence)")
            control_scale = gr.Slider(0.1, 2.0, value=0.55, step=0.05, label="ControlNet strength (structure)")

            with gr.Accordion("Canny parameters", open=False):
                canny_low = gr.Slider(0, 255, value=80, step=1, label="Canny low threshold")
                canny_high = gr.Slider(0, 255, value=200, step=1, label="Canny high threshold")

            with gr.Accordion("Color reference (IP-Adapter, optional)", open=False):
                use_ref = gr.Checkbox(value=False, label="Enable color reference (auto-disables if first load fails)")
                ref_image = gr.Image(type="pil", label="Color/style reference image (optional; defaults to control image)")
                ref_strength = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label="Reference strength (0=none, 1=strong)")

            run_btn = gr.Button("Generate", variant="primary")

    with gr.Row():
        out_image = gr.Image(label="Generated result", interactive=False)
        status = gr.Markdown()

    run_btn.click(
        fn=generate,
        inputs=[
            prompt,
            negative_prompt,
            control_image,
            height,
            width,
            steps,
            guidance,
            control_scale,
            canny_low,
            canny_high,
            seed,
            use_ref,
            ref_image,
            ref_strength,
        ],
        outputs=[out_image, status],
        queue=True,
    )

# Faster model download; Gradio queue API (limit queue size to 1)
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
demo.queue(max_size=1).launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3dba2c2579750519cb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from PIL import Image
import os

# ========== Left-right concatenation ==========
def align_height(a: Image.Image, b: Image.Image, pad: int = 60, bg=(255, 255, 255)) -> Image.Image:
    """
    Align two images by height and concatenate them side-by-side.

    Args:
        a (Image.Image): Left image.
        b (Image.Image): Right image.
        pad (int): Horizontal gap between the two images (pixels).
        bg (tuple[int, int, int]): Background color as (R, G, B).

    Returns:
        Image.Image: The concatenated image.
    """
    Ht = max(a.height, b.height)

    def to_h(x: Image.Image) -> Image.Image:
        if x.height == Ht:
            return x
        r = Ht / x.height
        return x.resize((int(x.width * r), Ht), Image.LANCZOS)

    A, B = to_h(a), to_h(b)
    out = Image.new("RGB", (A.width + pad + B.width, Ht), bg)
    out.paste(A, (0, 0))
    out.paste(B, (A.width + pad, 0))
    return out

# ========== Load two images ==========
left_img_path = "/content/WechatIMG122.jpg"            # Change to your path
right_img_path = "/content/owlvit-base-patch32_1.webp" # Change to your path
output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)

left_img = Image.open(left_img_path).convert("RGB")
right_img = Image.open(right_img_path).convert("RGB")

# ========== Concatenate & save ==========
side_by_side = align_height(left_img, right_img, pad=60)
side_by_side_path = os.path.join(output_dir, "side_by_side.jpg")
side_by_side.save(side_by_side_path)

print(f"Saved: {side_by_side_path}")


拼接完成: outputs/side_by_side.jpg


#owlvit-base-patch32

In [ ]:
!pip install -U transformers

In [ ]:
"""
OWL-ViT zero-shot detection (disaster-scene generalization) + multi-threshold + multi-scale + tiling + NMS + visualization/cropping/comparison
Outputs:
- outputs/detected_image.jpg
- outputs/side_by_side.jpg
- outputs/side_by_side_with_crops.jpg
"""

import os, math, random
from typing import List, Tuple
import torch
from PIL import Image, ImageDraw, ImageFont

from transformers import OwlViTProcessor, OwlViTForObjectDetection

# ===================== Configurable parameters =====================
image_path = "/content/WechatIMG122.jpg"   # ← change to your image path
output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, "crops"), exist_ok=True)

# Prompts (broad/abstract to improve recall; trim or replace with your own vocabulary)
texts = [[
    # Buildings/structures (generic + damaged)
    "building", "house", "structure", "infrastructure",
    "damaged building", "destroyed building", "collapsed building", "partially collapsed building",
    "damaged roof", "roof collapse", "missing roof", "cracked wall", "leaning structure",
    # Trees/vegetation (generic + fallen)
    "tree", "palm tree", "vegetation",
    "fallen tree", "downed tree", "uprooted tree", "broken tree", "tree debris", "damaged vegetation",
]]

# Visualization / post-processing
line_width = 4                  # box line width
title_font_size = 40
label_font_size = 20
crop_padding_ratio = 0.02       # crop padding as a fraction of the shorter side

# Stronger detection heuristics
thresholds = [0.25, 0.20, 0.15, 0.10, 0.07, 0.05, 0.03, 0.02, 0.01]  # progressively lower until hits are found
tile_cols = 6                  # number of horizontal tiles (use 4–6 for very wide images)
tile_overlap = 0.25            # overlap ratio between adjacent tiles (0.1–0.25)
multiscale_upscale = 2.0       # multi-scale: upscale tiles before detection (1.5–2.0)
nms_iou = 0.7                  # NMS IoU threshold
max_dets = 700                 # maximum detections to keep
crop_padding_ratio = 0.03      # crop padding ratio (overrides the value above)
# ==================================================================


# ===================== Utilities =====================

def get_font(size=20) -> ImageFont.FreeTypeFont:
    for fp in ["arial.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"]:
        try:
            return ImageFont.truetype(fp, size=size)
        except:
            pass
    return ImageFont.load_default()


title_font = get_font(title_font_size)
label_font = get_font(label_font_size)

def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def color_for(name: str) -> Tuple[int, int, int]:
    random.seed(hash(name) % (2**32))
    return tuple(random.randint(60, 230) for _ in range(3))


def add_title_bar(pil_img: Image.Image, title: str, pad=16, bar_h=None) -> Image.Image:
    w, h = pil_img.size
    if bar_h is None:
        bar_h = title_font_size + pad * 2
    canvas = Image.new("RGB", (w, h + bar_h), (255, 255, 255))
    canvas.paste(pil_img, (0, bar_h))
    d = ImageDraw.Draw(canvas)
    tw, th = d.textbbox((0, 0), title, font=title_font)[2:]
    d.text(((w - tw) // 2, (bar_h - th) // 2), title, fill=(0, 0, 0), font=title_font)
    return canvas


def align_height(a: Image.Image, b: Image.Image, pad=60, bg=(255, 255, 255)) -> Image.Image:
    Ht = max(a.height, b.height)

    def to_h(x):
        if x.height == Ht:
            return x
        r = Ht / x.height
        return x.resize((int(x.width * r), Ht), Image.LANCZOS)

    A, B = to_h(a), to_h(b)
    out = Image.new("RGB", (A.width + pad + B.width, Ht), bg)
    out.paste(A, (0, 0))
    out.paste(B, (A.width + pad, 0))
    return out


def make_grid(images: List[Image.Image], labels: List[str], cell_size=280, padding=14, bg=(245, 245, 245)) -> Image.Image:
    n = len(images)
    if n == 0:
        g = Image.new("RGB", (cell_size, cell_size), bg)
        d = ImageDraw.Draw(g)
        d.text((10, cell_size // 2 - 10), "No crops", fill=(0, 0, 0), font=get_font(20))
        return g
    cols = int(math.ceil(math.sqrt(n)))
    rows = int(math.ceil(n / cols))

    label_font_local = get_font(16)
    label_h = label_font_local.getbbox("Ay")[3] - label_font_local.getbbox("Ay")[1] + 6

    grid_w = cols * cell_size + (cols + 1) * padding
    grid_h = rows * (cell_size + label_h) + (rows + 1) * padding
    grid = Image.new("RGB", (grid_w, grid_h), bg)
    gdraw = ImageDraw.Draw(grid)

    def fit_to_square(img, size):
        w, h = img.size
        scale = min(size / w, size / h)
        new_w, new_h = int(w * scale), int(h * scale)
        return img.resize((new_w, new_h), Image.LANCZOS)

    for i, (im, lab) in enumerate(zip(images, labels)):
        r = i // cols
        c = i % cols
        x0 = padding + c * (cell_size + padding)
        y0 = padding + r * (cell_size + label_h + padding)

        fitted = fit_to_square(im, cell_size)
        fw, fh = fitted.size
        paste_x = x0 + (cell_size - fw) // 2
        paste_y = y0 + (cell_size - fh) // 2
        grid.paste(fitted, (paste_x, paste_y))

        show_lab = lab if len(lab) <= 28 else lab[:27] + "…"
        gdraw.text((x0, y0 + cell_size + 2), show_lab, fill=(0, 0, 0), font=label_font_local)
        gdraw.rectangle([x0 - 1, y0 - 1, x0 + cell_size + 1, y0 + cell_size + label_h + 1], outline=(220, 220, 220), width=1)

    return grid


# ===================== Inference functions  =====================

def load_model():
    processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
    model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device).eval()
    return processor, model, device


def run_model(processor, model, device, pil_img, text_prompts, thr):
    inputs = processor(text=text_prompts, images=pil_img, return_tensors="pt")
    inputs = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in inputs.items()}

    with torch.no_grad():
        outs = model(**inputs)  # ModelOutput

    for k, v in outs.items():
        if isinstance(v, torch.Tensor):
            outs[k] = v.detach().cpu()

    H_, W_ = pil_img.height, pil_img.width
    res = processor.post_process_grounded_object_detection(
        outputs=outs,  # stay as ModelOutput, not a dict
        target_sizes=torch.tensor([(H_, W_)]),
        threshold=thr,
        text_labels=text_prompts,
    )[0]
    return res


def nms_xyxy(boxes, scores, iou_thr=0.5):
    if len(boxes) == 0:
        return []
    try:
        from torchvision.ops import nms
        keep = nms(boxes, scores, iou_thr)
        return keep.tolist()
    except Exception:
        b = boxes.clone()
        x1, y1, x2, y2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
        areas = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
        order = torch.argsort(scores, descending=True)
        keep = []
        while order.numel() > 0:
            i = order[0].item()
            keep.append(i)
            if order.numel() == 1:
                break
            o = order[1:]
            xx1 = torch.maximum(x1[i], x1[o])
            yy1 = torch.maximum(y1[i], y1[o])
            xx2 = torch.minimum(x2[i], x2[o])
            yy2 = torch.minimum(y2[i], y2[o])
            inter = (xx2 - xx1).clamp(min=0) * (yy2 - yy1).clamp(min=0)
            iou = inter / (areas[i] + areas[o] - inter + 1e-6)
            order = o[iou <= iou_thr]
        return keep


def detect_pyramid_tiled(orig_img: Image.Image, text_prompts, processor, model, device):
    W, H = orig_img.size
    all_boxes, all_scores, all_labels = [], [], []

    # Compute horizontal tiling windows
    tile_w = W / tile_cols
    step = tile_w * (1 - tile_overlap)
    starts = [int(round(i * step)) for i in range(tile_cols)]
    if starts[-1] + tile_w < W:  # cover the rightmost edge
        starts[-1] = max(0, W - int(round(tile_w)))

    for col, x0 in enumerate(starts):
        x1 = int(round(min(W, x0 + tile_w)))
        crop = orig_img.crop((x0, 0, x1, H))

        found = False
        # 1) Multi-threshold
        for thr in thresholds:
            res = run_model(processor, model, device, crop, text_prompts, thr)
            if len(res["scores"]) > 0:
                use_res, use_thr = res, thr
                found = True
                break
        # 2) Multi-scale (upscale)
        if not found:
            big = crop.resize((int(crop.width * multiscale_upscale),
                               int(crop.height * multiscale_upscale)), Image.LANCZOS)
            for thr in thresholds:
                res2 = run_model(processor, model, device, big, text_prompts, thr)
                if len(res2["scores"]) > 0:
                    res2["boxes"] = res2["boxes"] / multiscale_upscale  # rescale back to tile coordinates
                    use_res, use_thr = res2, thr
                    found = True
                    break

        if found and len(use_res["scores"]) > 0:
            g_boxes = use_res["boxes"].clone()
            g_boxes[:, [0, 2]] += x0  # shift x-coordinates back to full image
            all_boxes.append(g_boxes)
            all_scores.append(use_res["scores"])
            all_labels += use_res["text_labels"]

    if len(all_boxes) == 0:
        return torch.empty((0, 4)), torch.empty((0,)), []

    boxes = torch.cat(all_boxes, dim=0)
    scores = torch.cat(all_scores, dim=0)

    # NMS & truncation
    keep = nms_xyxy(boxes, scores, iou_thr=nms_iou)
    keep = keep[:max_dets]
    boxes = boxes[keep]
    scores = scores[keep]
    labels = [all_labels[i] for i in keep]
    return boxes, scores, labels


# ===================== Main pipeline =====================

def main():
    # Load model
    processor, model, device = load_model()

    # Read image
    img = Image.open(image_path).convert("RGB")
    W, H = img.size

    # Detect (tiling + multi-threshold + multi-scale + NMS)
    boxes, scores, labels = detect_pyramid_tiled(img, texts, processor, model, device)

    # Visualization: original + detections
    det = img.copy()
    draw = ImageDraw.Draw(det)

    for box, score, name in zip(boxes, scores, labels):
        xmin, ymin, xmax, ymax = [float(v) for v in box.tolist()]
        xmin, ymin = clamp(xmin, 0, W - 1), clamp(ymin, 0, H - 1)
        xmax, ymax = clamp(xmax, 0, W - 1), clamp(ymax, 0, H - 1)

        col = color_for(name)
        draw.rectangle([(xmin, ymin), (xmax, ymax)], outline=col, width=line_width)

        txt = f"{name} {score:.2f}"
        tw, th = draw.textbbox((0, 0), txt, font=label_font)[2:]
        pad = 3
        y_text_top = max(0, ymin - th - 2 * pad)
        draw.rectangle([(xmin, y_text_top), (xmin + tw + 2 * pad, y_text_top + th + 2 * pad)], fill=col)
        draw.text((xmin + pad, y_text_top + pad), txt, fill=(255, 255, 255), font=label_font)

    detected_only_path = os.path.join(output_dir, "detected_image.jpg")
    det.save(detected_only_path)

    # Side-by-side: Original vs Detections
    left = add_title_bar(img, "Original image")
    right = add_title_bar(det, "Detections")
    side_by_side = align_height(left, right, pad=60)
    side_by_side_path = os.path.join(output_dir, "side_by_side.jpg")
    side_by_side.save(side_by_side_path)

    # Crops & grid
    short_side = min(W, H)
    pad_px = int(short_side * crop_padding_ratio)
    crops, crop_labels = [], []
    for idx, (box, score, name) in enumerate(zip(boxes, scores, labels), start=1):
        xmin, ymin, xmax, ymax = [int(round(v)) for v in box.tolist()]
        xmin = clamp(xmin - pad_px, 0, W)
        ymin = clamp(ymin - pad_px, 0, H)
        xmax = clamp(xmax + pad_px, 0, W)
        ymax = clamp(ymax + pad_px, 0, H)
        if xmax - xmin <= 2 or ymax - ymin <= 2:
            continue
        crop_img = img.crop((xmin, ymin, xmax, ymax))
        safe = name.replace(" ", "_").replace("/", "-")
        crop_img.save(os.path.join(output_dir, "crops", f"{idx:02d}_{safe}_{float(score):.2f}.jpg"))
        crops.append(crop_img)
        crop_labels.append(f"{idx:02d} {name} {float(score):.2f}")

    grid = make_grid(crops, crop_labels, cell_size=280, padding=14)
    grid_titled = add_title_bar(grid, "Crops grid")
    final_compare = align_height(add_title_bar(img, "Original image"), grid_titled, pad=60)
    final_compare_path = os.path.join(output_dir, "side_by_side_with_crops.jpg")
    final_compare.save(final_compare_path)

    print(f"Done (tiled={tile_cols}, overlap={tile_overlap}, up={multiscale_upscale}, NMS={nms_iou})")
    print(f"- Detections: {len(boxes)}")
    print(f"- Detections only: {detected_only_path}")
    print(f"- Original vs Detections: {side_by_side_path}")
    print(f"- Original vs Crops grid: {final_compare_path}")


if __name__ == "__main__":
    main()


完成（tiled=6, overlap=0.25, up=2.0, NMS=0.7）
- 命中数: 8
- 仅检测可视化: outputs/detected_image.jpg
- 原图 vs 检测: outputs/side_by_side.jpg
- 原图 vs 裁剪网格: outputs/side_by_side_with_crops.jpg
